# Webカメラ定期自動撮影（インターバル監視）眠気検知システム

Google Colaboratory上でWebカメラを起動し、**3秒おきに自動で静止画を撮影してEARによる眠気判定**を行います。

---
### 🌟 2つの実装アプローチ
1. **【対策C】単一セル完結型 (`update_display` 方式) ★推奨**
   - `clear_output()` を使わず、画像の表示枠（DisplayHandle）のみをインプレースで更新します。
   - カメラDOMが破棄されないため、**1つのセルを実行するだけで起動から連続監視・停止まで完結**します。
2. **【対策B】セル分割型**
   - カメラ起動セルと判定ループセルを物理的に分離する方式です。

In [ ]:
# [1] ライブラリのインストールとインポート
!pip install face_recognition

import base64
import io
import time
import cv2
import face_recognition
import numpy as np
from PIL import Image
from IPython.display import display, Javascript, HTML, clear_output, Image as IPyImage
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow

print("✅ ライブラリの準備が完了しました。")

In [ ]:
# [2] 共通関数の定義（Webカメラ制御・画像変換・EAR計算）

def start_webcam_stream():
    """
    Webカメラを起動し、ブラウザ上にプレビュー表示およびストリームを保持する。
    """
    js = Javascript('''
        async function startStream() {
            if (window.colabStream) {
                return;
            }
            const container = document.createElement('div');
            container.id = 'camera_monitor_container';
            container.style.fontFamily = 'Arial, sans-serif';
            container.style.marginBottom = '12px';
            container.style.padding = '8px';
            container.style.border = '2px solid #1a73e8';
            container.style.borderRadius = '8px';
            container.style.width = 'max-content';
            container.style.backgroundColor = '#f8f9fa';

            const title = document.createElement('div');
            title.innerHTML = '📷 <b>定期監視Webカメラ稼働中</b> (バックグラウンド待機中)';
            title.style.marginBottom = '6px';
            title.style.color = '#1a73e8';
            title.style.fontSize = '14px';
            container.appendChild(title);

            const video = document.createElement('video');
            video.id = 'camera_monitor_video';
            video.width = 320;
            video.height = 240;
            video.style.display = 'block';
            video.style.borderRadius = '6px';
            video.style.transform = 'scaleX(-1)'; // 鏡像表示

            const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
            video.srcObject = stream;
            container.appendChild(video);
            document.body.appendChild(container);

            window.colabStream = stream;
            window.colabVideo = video;
            window.colabContainer = container;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        }
        startStream();
    ''')
    display(js)

def capture_frame(quality=0.8):
    """
    起動中のWebカメラから最新フレームを1枚取得し、OpenCV(BGR)画像として返す。
    """
    data = eval_js(f'''
        (function() {{
            const video = window.colabVideo;
            if (!video) return null;
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth || 640;
            canvas.height = video.videoHeight || 480;
            const ctx = canvas.getContext('2d');
            ctx.translate(canvas.width, 0);
            ctx.scale(-1, 1);
            ctx.drawImage(video, 0, 0);
            return canvas.toDataURL('image/jpeg', {quality});
        }})()
    ''')
    if not data:
        return None

    header, encoded = data.split(',', 1)
    binary = base64.b64decode(encoded)
    image_np = np.frombuffer(binary, dtype=np.uint8)
    image_bgr = cv2.imdecode(image_np, cv2.IMREAD_COLOR)
    return image_bgr

def stop_webcam():
    """
    Webカメラストリームを停止し、UIを削除してリソースを解放する。
    """
    js = Javascript('''
        function stopCamera() {
            if (window.colabStream) {
                window.colabStream.getVideoTracks().forEach(track => track.stop());
                window.colabStream = null;
            }
            if (window.colabContainer) {
                window.colabContainer.remove();
                window.colabContainer = null;
                window.colabVideo = null;
            }
        }
        stopCamera();
    ''')
    display(js)

def calculate_ear(eye_landmarks):
    """
    目の周りの6点座標からEAR(Eye Aspect Ratio)を計算する。
    """
    p2_p6 = np.linalg.norm(np.array(eye_landmarks[1]) - np.array(eye_landmarks[5]))
    p3_p5 = np.linalg.norm(np.array(eye_landmarks[2]) - np.array(eye_landmarks[4]))
    p1_p4 = np.linalg.norm(np.array(eye_landmarks[0]) - np.array(eye_landmarks[3]))
    if p1_p4 == 0:
        return 0.0
    return (p2_p6 + p3_p5) / (2.0 * p1_p4)

def cv2_to_ipyimage(img_bgr, quality=85):
    """
    OpenCV(BGR)画像をIPython.display.Imageオブジェクトに変換する（インプレース描画用）
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=quality)
    return IPyImage(data=buf.getvalue())

print("✅ 関数の定義が完了しました。")

## 🚀 【対策C】単一セル完結型インターバル監視（update_display方式）

`clear_output()` を使用せず、`display(..., display_id=True)` のハンドルに対して **インプレース更新（`update`）** を行います。
カメラDOMが消滅しないため、**このセル1つを実行するだけで自動監視が開始されます。**

In [ ]:
# [対策C] 単一セル完結型 定期自動撮影 & EAR眠気判定

INTERVAL_SECONDS = 3       # 撮影間隔（秒）
EAR_THRESHOLD = 0.22       # 眠気判定の閾値
SLEEPY_STREAK_ALERT = 2    # 連続何回「眠気」判定で警告を出すか

# 1. Webカメラの起動
print("📷 Webカメラを起動しています...")
start_webcam_stream()
time.sleep(1.5) # カメラ安定待ち

# 2. 表示更新用ハンドル（DisplayHandle）の作成
text_handle = display(HTML("<div style='font-family: sans-serif; padding: 6px; color: #1a73e8;'>⚡ 監視待機中...</div>"), display_id="status_text_display")
image_handle = display(HTML("<div style='color: #666;'>画像を読み込んでいます...</div>"), display_id="result_image_display")

iteration = 0
sleepy_count = 0

try:
    while True:
        iteration += 1
        current_time_str = time.strftime('%H:%M:%S')

        # 最新フレームの取得
        frame_bgr = capture_frame()
        if frame_bgr is None:
            time.sleep(1)
            continue

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        landmarks_list = face_recognition.face_landmarks(frame_rgb)

        if len(landmarks_list) == 0:
            sleepy_count = 0
            status_text = "Status: NO FACE DETECTED"
            badge_html = "<span style='background:#fbbc04; color:#202124; padding:3px 8px; border-radius:4px; font-weight:bold;'>⚠️ 顔未検出</span>"
        else:
            landmarks = landmarks_list[0]
            left_eye = landmarks['left_eye']
            right_eye = landmarks['right_eye']

            left_ear = calculate_ear(left_eye)
            right_ear = calculate_ear(right_eye)
            avg_ear = (left_ear + right_ear) / 2.0

            # 目の輪郭描画
            for eye in [left_eye, right_eye]:
                pts = np.array(eye, np.int32)
                cv2.polylines(frame_bgr, [pts], isClosed=True, color=(255, 200, 0), thickness=2)

            # 眠気判定
            if avg_ear < EAR_THRESHOLD:
                sleepy_count += 1
                if sleepy_count >= SLEEPY_STREAK_ALERT:
                    status_text = f"!! DROWSINESS ALERT !! (EAR: {avg_ear:.2f})"
                    status_color = (0, 0, 255)
                    cv2.rectangle(frame_bgr, (0, 0), (frame_bgr.shape[1]-1, frame_bgr.shape[0]-1), (0, 0, 255), 6)
                    badge_html = f"<span style='background:#ea4335; color:#fff; padding:3px 8px; border-radius:4px; font-weight:bold;'>🚨 居眠り警告 (連続 {sleepy_count} 回)</span>"
                else:
                    status_text = f"Status: SLEEPY (EAR: {avg_ear:.2f})"
                    status_color = (0, 128, 255)
                    badge_html = f"<span style='background:#ff9800; color:#fff; padding:3px 8px; border-radius:4px; font-weight:bold;'>⚠️ 眠気検知 (EAR: {avg_ear:.3f})</span>"
            else:
                sleepy_count = 0
                status_text = f"Status: AWAKE (EAR: {avg_ear:.2f})"
                status_color = (0, 255, 0)
                badge_html = f"<span style='background:#34a853; color:#fff; padding:3px 8px; border-radius:4px; font-weight:bold;'>✅ 覚醒中 (EAR: {avg_ear:.3f})</span>"

            # 結果バナー描画
            cv2.rectangle(frame_bgr, (15, 15), (460, 75), (30, 30, 30), -1)
            cv2.putText(frame_bgr, status_text, (25, 52), cv2.FONT_HERSHEY_SIMPLEX, 0.8, status_color, 2, cv2.LINE_AA)
            cv2.putText(frame_bgr, f"Avg EAR: {avg_ear:.3f} | Sleepy Streak: {sleepy_count}", (25, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220, 220, 220), 1, cv2.LINE_AA)

        # 3. clear_output() を使わずに、DisplayHandleをその場で更新！
        info_html = f"""
        <div style='font-family: Arial, sans-serif; background: #202124; color: #fff; padding: 8px 12px; border-radius: 6px; margin: 8px 0; max-width: 600px;'>
            <div style='display: flex; justify-content: space-between; align-items: center;'>
                <span>🕒 <b>サイクル #{iteration}</b> ({current_time_str} / {INTERVAL_SECONDS}秒間隔)</span>
                <span>{badge_html}</span>
            </div>
            <div style='font-size: 12px; color: #aaa; margin-top: 4px;'>※ 停止するにはセル左側の「■ (実行停止)」ボタンを押してください。</div>
        </div>
        """
        text_handle.update(HTML(info_html))
        image_handle.update(cv2_to_ipyimage(frame_bgr))

        time.sleep(INTERVAL_SECONDS)

except KeyboardInterrupt:
    print("\n🛑 ユーザー操作により定期監視ループを停止しました。")
finally:
    stop_webcam()
    print("✅ Webカメラの接続を解除し、リソースを正常に解放しました。")